In [ ]:
!uv pip install bitsandbytes trl

Using Python 3.12.13 environment at: /usr
Resolved 78 packages in 575ms
Prepared 4 packages in 1.13s
Uninstalled 2 packages in 39ms
Installed 4 packages in 15ms
 + bitsandbytes==0.49.2
 - datasets==4.0.0
 + datasets==4.8.5
 - pyarrow==18.1.0
 + pyarrow==24.0.0
 + trl==1.3.0


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(torch.cuda.get_device_name(0))
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [ ]:
import os
import torch, time, random, json, re
import numpy as np
from datasets import load_dataset
from transformers import TrainingArguments, default_data_collator, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
class Config:
    SEED = 42
    MODEL_NAME = "mistralai/Mistral-7B-v0.1"

    # Dataset Sizes
    TRAIN_SIZE = 2000
    TEST_SIZE = 200
    GEN_SIZE = 100
    FORGET_SIZE = 100

    # Training Params
    MAX_SEQ_LEN = 512
    CONTEXT_LEN = 800
    EPOCHS = 2
    BATCH_SIZE = 4
    LR = 2e-4

    # LoRA Params
    LORA_R = 8
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.05

    # Paths
    OUTPUT_DIR = "./lora_model"
    RESULTS_FILE = "results.json"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)

In [ ]:
def load_squad_splits():
    print("Loading SQuAD Dataset splits...")
    return {
        'train': load_dataset("squad", split=f"train[:{Config.TRAIN_SIZE}]"),
        'test': load_dataset("squad", split=f"validation[:{Config.TEST_SIZE}]"),
        'gen': load_dataset("squad", split=f"validation[{Config.TEST_SIZE}:{Config.TEST_SIZE + Config.GEN_SIZE}]"),
        'forget': load_dataset("squad", split=f"validation[{Config.TEST_SIZE + Config.GEN_SIZE}:{Config.TEST_SIZE + Config.GEN_SIZE + Config.FORGET_SIZE}]")
    }

datasets = load_squad_splits()
for k, v in datasets.items():
    print(f"{k.capitalize():<10}: {len(v)} samples")

Loading SQuAD Dataset splits...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Train     : 2000 samples
Test      : 200 samples
Gen       : 100 samples
Forget    : 100 samples


In [ ]:
from peft import AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    Config.MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# AdaLoRA requires total_step to schedule rank budget
# Total steps = (Train samples / (Batch size * Grad accumulation)) * Epochs
total_steps = (len(datasets['train']) // (Config.BATCH_SIZE * 4)) * Config.EPOCHS

adalora_config = AdaLoraConfig(
    peft_type="ADALORA",
    task_type="CAUSAL_LM",
    r=Config.LORA_R,
    lora_alpha=Config.LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=Config.LORA_DROPOUT,
    target_r=8,
    init_r=12,
    tinit=200,
    tfinal=1000,
    deltaT=10,
    beta1=0.85,
    beta2=0.85,
    orth_reg_weight=0.5,
    total_step=max(total_steps, 1500) # Ensure it's large enough for the tinit/tfinal schedule
)

lora_model = get_peft_model(model, adalora_config)
lora_model.print_trainable_parameters()
print(f"AdaLoRA model initialized with total_step={total_steps}.")

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

trainable params: 10,225,152 || all params: 7,251,957,376 || trainable%: 0.1410
AdaLoRA model initialized with total_step=250.


In [ ]:
def tokenize_fn(example):
    prompt = f"Context: {example['context'][:Config.CONTEXT_LEN]}\nQuestion: {example['question']}\nAnswer: "
    full_text = prompt + example['answers']['text'][0]

    tokens = tokenizer(full_text, truncation=True, max_length=Config.MAX_SEQ_LEN, padding="max_length")
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=Config.MAX_SEQ_LEN)['input_ids'])

    labels = ([-100] * prompt_len + tokens["input_ids"][prompt_len:])[:Config.MAX_SEQ_LEN]
    labels += [-100] * (Config.MAX_SEQ_LEN - len(labels))
    tokens["labels"] = labels
    return tokens

def normalize_text(text):
    text = text.strip().lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def get_metrics(pred, gold):
    pred, gold = normalize_text(pred), normalize_text(gold)
    em = 1 if pred == gold else 0
    p_tokens, g_tokens = pred.split(), gold.split()
    if not p_tokens or not g_tokens: return em, 0.0
    common = set(p_tokens) & set(g_tokens)
    if not common: return em, 0.0
    prec, rec = len(common)/len(p_tokens), len(common)/len(g_tokens)
    f1 = 2 * prec * rec / (prec + rec)
    return em, f1

def run_evaluation(model, dataset, name="Eval"):
    model.eval()
    results = {'em': 0, 'f1': 0, 'acc': 0, 'time': 0}
    print(f"\nEvaluating {name}...")

    for i, item in enumerate(dataset):
        prompt = f"Context: {item['context'][:Config.CONTEXT_LEN]}\nQuestion: {item['question']}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=Config.MAX_SEQ_LEN).to("cuda")

        t0 = time.time()
        with torch.no_grad():
            # Explicitly set pad_token_id to suppress warnings
            out = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id
            )
        results['time'] += (time.time() - t0)

        gen = tokenizer.decode(out[0], skip_special_tokens=True)
        pred = gen.split("Answer:")[-1].split("\n")[0].strip().lower()
        gold = item['answers']['text'][0].lower()

        em, f1 = get_metrics(pred, gold)
        results['em'] += em
        results['f1'] += f1
        if gold in pred or pred in gold: results['acc'] += 1

    count = len(dataset)
    return {k: (v/count)*100 if k != 'time' else v/count for k, v in results.items()}

train_tokenized = datasets['train'].map(tokenize_fn, remove_columns=datasets['train'].column_names)
test_tokenized = datasets['test'].map(tokenize_fn, remove_columns=datasets['test'].column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
base_results = {
    'main': run_evaluation(model, datasets['test'], "Base-Main"),
    'gen': run_evaluation(model, datasets['gen'], "Base-Generalization"),
    'forget': run_evaluation(model, datasets['forget'], "Base-Forgetting")
}

# We'll keep the model in memory for LoRA initialization next rather than deleting it
torch.cuda.empty_cache()


Evaluating Base-Main...

Evaluating Base-Generalization...

Evaluating Base-Forgetting...


In [ ]:
base_results

{'main': {'em': 57.99999999999999,
  'f1': 68.70120651192633,
  'acc': 73.0,
  'time': 2.6095007956027985},
 'gen': {'em': 32.0,
  'f1': 37.49975457119419,
  'acc': 46.0,
  'time': 3.306252143383026},
 'forget': {'em': 59.0,
  'f1': 69.20966497270844,
  'acc': 75.0,
  'time': 3.120936031341553}}

In [ ]:
def formatting_prompts_func(example):
    # For SFTTrainer, return a single string for each example
    return f"Context: {example['context'][:Config.CONTEXT_LEN]}\nQuestion: {example['question']}\nAnswer: {example['answers']['text'][0]}"

trainer = SFTTrainer(
    model=lora_model,
    train_dataset=datasets['train'],
    eval_dataset=datasets['test'],
    formatting_func=formatting_prompts_func,
    args=SFTConfig(
    output_dir=Config.OUTPUT_DIR,
    num_train_epochs=Config.EPOCHS,
    per_device_train_batch_size=Config.BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=Config.LR,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,   # ← turn off
    bf16=False,   # ← turn off too
    optim="adamw_8bit",
    report_to="none",
    seed=Config.SEED,
    remove_unused_columns=False
),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting AdaLoRA training...")
start_time = time.time()
trainer.train()
train_duration = (time.time() - start_time) / 60
lora_model.save_pretrained(Config.OUTPUT_DIR)

Applying formatting function to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting AdaLoRA training...


Epoch,Training Loss,Validation Loss
1,2.403592,1.304197
2,1.579199,1.243033


In [ ]:
print("=== STEP 7: AdaLoRA EVALUATION ===")
adalora_results = {
    'main': run_evaluation(lora_model, datasets['test'], "AdaLoRA-Main"),
    'gen': run_evaluation(lora_model, datasets['gen'], "AdaLoRA-Generalization"),
    'forget': run_evaluation(lora_model, datasets['forget'], "AdaLoRA-Forgetting")
}

=== STEP 7: AdaLoRA EVALUATION ===

Evaluating AdaLoRA-Main...

Evaluating AdaLoRA-Generalization...

Evaluating AdaLoRA-Forgetting...


In [ ]:
print("\n" + "="*60)
print("FINAL REFACTORED SUMMARY")
print("="*60)
print(f"Base Acc: {base_results['main']['acc']:.1f}% | LoRA Acc: {adalora_results['main']['acc']:.1f}%")
print(f"Base F1 : {base_results['main']['f1']:.1f}% | LoRA F1 : {adalora_results['main']['f1']:.1f}%")
print(f"Generalization Delta: {adalora_results['gen']['acc'] - base_results['gen']['acc']:+.1f}%")
print(f"Forgetting Delta: {adalora_results['forget']['acc'] - base_results['forget']['acc']:+.1f}%")
print(f"Training Time: {train_duration:.1f} min")
print(f"AdaLoRA Average Latency: {adalora_results['main']['time']:.4f} seconds/example")
print(f"AdaLoRA EM: {adalora_results['main']['em']:.1f}%")


with open(Config.RESULTS_FILE, "w") as f:
    json.dump({'base': base_results, 'lora': adalora_results}, f, indent=2)


FINAL REFACTORED SUMMARY
Base Acc: 73.0% | LoRA Acc: 88.0%
Base F1 : 68.7% | LoRA F1 : 89.7%
Generalization Delta: +31.0%
Forgetting Delta: +15.0%
Training Time: 12.2 min
AdaLoRA Average Latency: 0.5832 seconds/example
AdaLoRA EM: 86.0%
